# Proyecto de Boosting - Prediccion de Diabetes

Este notebook aplica la solucion oficial del proyecto de Mejorando Algoritmos usando boosting para mejorar la prediccion de diabetes.

La solucion oficial entrena un modelo:

```python
XGBClassifier(n_estimators=200, learning_rate=0.001, random_state=42)
```

Ademas, aqui se conserva la comparacion solicitada por el enunciado entre modelos previos y boosting.

## 1. Importacion de librerias y funciones

El codigo productivo esta en `src/app.py`. Desde este notebook importamos el pipeline para mantener el flujo reproducible.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from app import (
    BEST_MODEL_PATH,
    BOOSTING_PLOT_PATH,
    CLASS_PRECISION_PATH,
    MODEL_COMPARISON_PATH,
    OFFICIAL_MODEL_PATH,
    TARGET_COLUMN,
    run_full_pipeline,
)

## 2. Carga de datos procesados

El enunciado pide reutilizar los datos procesados del proyecto anterior. Por eso el proyecto trabaja con:

- `data/processed/clean_train.csv`
- `data/processed/clean_test.csv`

Si esos archivos no existieran, `app.py` puede recrearlos desde `data/raw/diabetes.csv` o desde el enlace del dataset.

In [2]:
outputs = run_full_pipeline(verbose=False)

train_df = outputs["train_df"]
test_df = outputs["test_df"]
comparison_df = outputs["comparison_df"].reset_index(drop=True)
precision_df = outputs["precision_df"].reset_index(drop=True)
boosting_results_df = outputs["boosting_results_df"]
reports = outputs["reports"]

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
display(train_df.head())

Aviso: XGBoost no pudo cargarse en este entorno local. En Codespaces deberia funcionar con la dependencia xgboost. Se usa GradientBoostingClassifier como fallback local. Detalle: No module named 'xgboost'


Train shape: (614, 9)
Test shape: (154, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,1,90.0,62.0,12.0,43.0,27.2,0.580,24,0
1,5,126.0,78.0,27.0,22.0,29.6,0.439,40,0
2,2,105.0,58.0,40.0,94.0,34.9,0.225,25,0
3,1,146.0,56.0,29.0,125.0,29.7,0.564,29,0
4,0,95.0,64.0,39.0,105.0,44.6,0.366,22,0


## 3. Comprobacion rapida del dataset

Se revisa que no haya nulos y que la proporcion de positivos sea parecida en train y test.

In [3]:
display(pd.DataFrame({
    "missing_train": train_df.isna().sum(),
    "missing_test": test_df.isna().sum(),
}))

summary = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "columns": [train_df.shape[1], test_df.shape[1]],
    "positive_rate": [train_df[TARGET_COLUMN].mean(), test_df[TARGET_COLUMN].mean()],
})
display(summary)

,missing_train,missing_test
Pregnancies,0,0
Glucose,0,0
BloodPressure,0,0
SkinThickness,0,0
Insulin,0,0
BMI,0,0
DiabetesPedigreeFunction,0,0
Age,0,0
Outcome,0,0


,dataset,rows,columns,positive_rate
0,train,614,9,0.348534
1,test,154,9,0.350649


## 4. Modelo de boosting de la solucion oficial

La solucion oficial usa `XGBClassifier` con estos hiperparametros:

- `n_estimators=200`
- `learning_rate=0.001`
- `random_state=42`

En este equipo local, XGBoost puede fallar si falta `libomp` en macOS. El script incluye un fallback local para poder ejecutar la validacion, pero en Codespaces usara XGBoost al instalar `xgboost` desde `requirements.txt`.

In [4]:
official_report = reports["official_xgboost_solution"]
print("Reporte del modelo oficial de boosting:")
display(pd.DataFrame(official_report).T)

Reporte del modelo oficial de boosting:


,precision,recall,f1-score,support
0,0.649351,1.000000,0.787402,100.000000
1,0.000000,0.000000,0.000000,54.000000
accuracy,0.649351,0.649351,0.649351,0.649351
macro avg,0.324675,0.500000,0.393701,154.000000
weighted avg,0.421656,0.649351,0.511300,154.000000


## 5. Ajuste adicional de hiperparametros

El enunciado tambien pide modificar hiperparametros y analizar su impacto. Para eso se ejecuta una busqueda sobre un modelo de boosting de scikit-learn, guardando los resultados en `data/interim/boosting_grid_results.csv`.

In [5]:
boosting_results_df[
    [
        "mean_test_score",
        "std_test_score",
        "param_n_estimators",
        "param_learning_rate",
        "param_max_depth",
        "param_subsample",
    ]
].head(10)

,mean_test_score,std_test_score,param_n_estimators,param_learning_rate,param_max_depth,param_subsample
20,0.786625,0.013135,100,0.05,2,0.8
37,0.785007,0.004429,50,0.10,2,1.0
26,0.781763,0.012084,100,0.05,3,0.8
10,0.781755,0.006200,200,0.01,3,0.8
21,0.781739,0.008806,100,0.05,2,1.0
16,0.780137,0.003562,200,0.01,4,0.8
24,0.780129,0.000507,50,0.05,3,0.8
25,0.780121,0.007172,50,0.05,3,1.0
51,0.778439,0.027115,100,0.10,4,1.0
54,0.775235,0.010723,50,0.20,2,0.8


## 6. Comparacion de modelos

Se comparan los enfoques principales del recorrido del modulo:

- Arbol de decision
- Random forest
- Boosting oficial con XGBoost
- Boosting ajustado para estudiar hiperparametros

La metrica principal del enunciado es `accuracy`.

In [6]:
display(comparison_df)

best_model = comparison_df.iloc[0]
print(f"Mejor modelo por accuracy: {best_model['model']} ({best_model['accuracy']:.4f})")

,model,accuracy,precision_weighted,recall_weighted,f1_weighted,class_0_precision,class_1_precision,best_precision_class,worst_precision_class
0,random_forest,0.740260,0.732375,0.740260,0.731377,0.767857,0.666667,0,1
1,tuned_gradient_boosting,0.733766,0.726391,0.733766,0.727397,0.770642,0.644444,0,1
2,decision_tree,0.688312,0.675718,0.688312,0.644189,0.696970,0.636364,0,1
3,official_xgboost_solution,0.649351,0.421656,0.649351,0.511300,0.649351,0.000000,0,1


Mejor modelo por accuracy: random_forest (0.7403)


## 7. Clase con mayor y menor precision

El enunciado pide analizar que clase se predice mejor y peor. Aqui se comparan las precisiones para la clase `0` y la clase `1`.

In [7]:
display(precision_df)

for _, row in comparison_df.iterrows():
    print(
        f"{row['model']}: mejor clase={row['best_precision_class']}, "
        f"peor clase={row['worst_precision_class']}"
    )

,model,precision_class_0,precision_class_1
0,decision_tree,0.696970,0.636364
1,official_xgboost_solution,0.649351,0.000000
2,random_forest,0.767857,0.666667
3,tuned_gradient_boosting,0.770642,0.644444


random_forest: mejor clase=0, peor clase=1
tuned_gradient_boosting: mejor clase=0, peor clase=1
decision_tree: mejor clase=0, peor clase=1
official_xgboost_solution: mejor clase=0, peor clase=1


## 8. Grafica de impacto de hiperparametros

La grafica generada por `app.py` resume como cambian los resultados al variar `learning_rate`, `n_estimators` y `max_depth`.

In [8]:
print("Grafica guardada en:", BOOSTING_PLOT_PATH)
BOOSTING_PLOT_PATH

Grafica guardada en: /Users/dragcessa/Desktop/bootcam 4gees/01_machine_learning/Dragcessa1998-Proyecto-Tutorial-de-Mejorando-Algoritmos-main 2/reports/figures/boosting_hyperparameter_impact.png


PosixPath('/Users/dragcessa/Desktop/bootcam 4gees/01_machine_learning/Dragcessa1998-Proyecto-Tutorial-de-Mejorando-Algoritmos-main 2/reports/figures/boosting_hyperparameter_impact.png')

## 9. Guardado del modelo

El modelo oficial queda guardado en dos rutas:

- Ruta compatible con la solucion oficial: `models/boosting_classifier_nestimators-20_learnrate-0.001_42.sav`
- Ruta descriptiva: `models/xgboost_diabetes_model.sav`

In [9]:
print("Modelo compatible con solucion oficial:", OFFICIAL_MODEL_PATH)
print("Modelo descriptivo:", BEST_MODEL_PATH)
print("Comparacion guardada en:", MODEL_COMPARISON_PATH)
print("Precision por clase guardada en:", CLASS_PRECISION_PATH)

Modelo compatible con solucion oficial: /Users/dragcessa/Desktop/bootcam 4gees/01_machine_learning/Dragcessa1998-Proyecto-Tutorial-de-Mejorando-Algoritmos-main 2/models/boosting_classifier_nestimators-20_learnrate-0.001_42.sav
Modelo descriptivo: /Users/dragcessa/Desktop/bootcam 4gees/01_machine_learning/Dragcessa1998-Proyecto-Tutorial-de-Mejorando-Algoritmos-main 2/models/xgboost_diabetes_model.sav
Comparacion guardada en: /Users/dragcessa/Desktop/bootcam 4gees/01_machine_learning/Dragcessa1998-Proyecto-Tutorial-de-Mejorando-Algoritmos-main 2/data/processed/model_comparison.csv
Precision por clase guardada en: /Users/dragcessa/Desktop/bootcam 4gees/01_machine_learning/Dragcessa1998-Proyecto-Tutorial-de-Mejorando-Algoritmos-main 2/data/processed/class_precision_comparison.csv


## 10. Conclusion

El modelo de boosting se implemento siguiendo la solucion oficial con `XGBClassifier`. Despues se comparo contra arbol de decision y random forest para cumplir el analisis final del ejercicio.

La seleccion final debe hacerse revisando `accuracy` y tambien la precision por clase, especialmente porque en problemas medicos puede interesar detectar mejor la clase positiva de diabetes.